# Generator-Solver Interaction Analysis

Investigates whether specific generators produce adaptations that are systematically easier or harder for specific solver models.

**Key questions:**
1. Do Gemini-generated adaptations help all models or just Gemini?
2. Is there a generator that produces universally easier/harder adaptations?
3. Are there specific generator-solver pairings that outperform or underperform?
4. Does domain choice by the generator correlate with solver accuracy?

## 1. Imports & Config

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy.stats import ttest_ind, mannwhitneyu

# ── Paths ─────────────────────────────────────────────────────────────────────
BENCHMARK_AUDITED  = '/content/all_models_audited_final.csv'
RESULTS_COMBINED   = '/content/results_combined_audited.csv'
ORIGINAL_PUZZLES   = '/content/puzzle_samples.xlsx'
OUTDIR             = './outputs/generator_solver'
os.makedirs(OUTDIR, exist_ok=True)

## 2. Load & Prepare Data

In [ ]:
# ── Benchmark solve results ───────────────────────────────────────────────────
bench = pd.read_csv(BENCHMARK_AUDITED)

# Drop Solved_Percentile if already present from a previous merge
bench = bench.drop(columns=['Solved_Percentile', 'difficulty'], errors='ignore')

# Strip instruction prefix to match adaptation text
prefix = 'Your task is to provide the correct answer to the given puzzle using the clues.\n'
bench['Sample_clean'] = bench['Sample'].str.replace(prefix, '', regex=False).str.strip()

# ── Adaptation results with generator info ────────────────────────────────────
adapt = pd.read_csv(RESULTS_COMBINED)[[
    'Puzzle_Number', 'Question_Number', 'generator_model',
    'final_adapted_puzzle', 'final_target_domain', 'corrected_outcome'
]]

# Only keep judge-confirmed adaptations
adapt_clean = adapt[adapt['corrected_outcome'].isin(['equivalent', 'rescued_formatting'])].copy()

# ── Merge: each benchmark solve row gets its generator_model + domain ─────────
merged = bench.merge(
    adapt_clean,
    left_on=['Puzzle_Number', 'Question_Number', 'Sample_clean'],
    right_on=['Puzzle_Number', 'Question_Number', 'final_adapted_puzzle'],
    how='left'
).drop(columns=['final_adapted_puzzle'])

# ── Original puzzle difficulty (same difficulty assumed for adapted puzzles) ───
orig = pd.read_excel(ORIGINAL_PUZZLES)[[
    'Puzzle_Number', 'Question_Number', 'Solved_Percentile'
]].drop_duplicates(subset=['Puzzle_Number', 'Question_Number'])

merged = merged.merge(orig, on=['Puzzle_Number', 'Question_Number'], how='left')

def map_difficulty(p):
    if p < 30:   return 'Hard'
    elif p < 60: return 'Medium'
    else:        return 'Easy'

merged['difficulty'] = merged['Solved_Percentile'].apply(map_difficulty)

print(f'Total benchmark rows      : {len(bench)}')
print(f'Rows with generator info  : {merged["generator_model"].notna().sum()}')
print(f'\nSolver models: {sorted(merged["model_name"].unique())}')
print(f'Generator models: {sorted(merged["generator_model"].dropna().unique())}')
print(f'\nGenerator breakdown:')
print(merged['generator_model'].value_counts().to_string())

In [ ]:
merged_pairs = set(zip(merged['Puzzle_Number'], merged['Question_Number']))
orig_pairs   = set(zip(orig['Puzzle_Number'],   orig['Question_Number']))
print(f'Merged pairs   : {len(merged_pairs)}')
print(f'Orig pairs     : {len(orig_pairs)}')
print(f'Intersection   : {len(merged_pairs & orig_pairs)}')
print(f'Sample merged  : {list(merged_pairs)[:5]}')
print(f'Sample orig    : {list(orig_pairs)[:5]}')

## 3. Generator × Solver Accuracy Matrix

In [ ]:
# Pivot: rows = solver models, cols = generator models
matrix = merged.groupby(['model_name', 'generator_model'])['correct'].mean().unstack() * 100

# Add row/col means
matrix['avg_across_generators'] = matrix.mean(axis=1)
matrix.loc['avg_across_solvers'] = matrix.mean(axis=0)

print('Generator × Solver Accuracy Matrix (%):')
print(matrix.round(1).to_string())
matrix.to_csv(os.path.join(OUTDIR, 'generator_solver_matrix.csv'))

## 4. Heatmap — Generator × Solver

In [ ]:
# Plot without the avg row/col for cleaner viz
plot_matrix = merged.groupby(['model_name', 'generator_model'])['correct'].mean().unstack() * 100

fig, ax = plt.subplots(figsize=(12, 6))
im = sns.heatmap(
    plot_matrix,
    annot=True, fmt='.1f',
    cmap='Blues',
    vmin=0, vmax=100,
    linewidths=0.5, linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Accuracy (%)'}
)
ax.set_xlabel('Generator Model', fontweight='bold', fontsize=11)
ax.set_ylabel('Solver Model', fontweight='bold', fontsize=11)
ax.set_title('Solver Accuracy by Generator × Solver Pair\n(% correct on adapted puzzles)',
             fontweight='bold', fontsize=13)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'heatmap_generator_solver.png'), dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Does Any Generator Help All Models?:

In [ ]:
# For each generator: how does each solver perform vs their average across all generators
solver_baselines = merged.groupby('model_name')['correct'].mean() * 100

rows = []
for gen, g_gen in merged.groupby('generator_model'):
    for solver, g_solver in g_gen.groupby('model_name'):
        acc        = g_solver['correct'].mean() * 100
        baseline   = solver_baselines[solver]
        delta      = acc - baseline
        rows.append({
            'generator': gen,
            'solver':    solver,
            'acc':       acc,
            'baseline':  baseline,
            'delta':     delta,
            'n':         len(g_solver),
        })

delta_df = pd.DataFrame(rows)

# Average delta per generator across all solvers
gen_avg_delta = delta_df.groupby('generator')['delta'].mean().sort_values(ascending=False)
print('Average delta vs solver baseline (positive = generator makes puzzles easier):')
print(gen_avg_delta.round(2).to_string())

# Plot delta heatmap
delta_pivot = delta_df.pivot(index='solver', columns='generator', values='delta')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    delta_pivot,
    annot=True, fmt='.1f',
    cmap='RdYlGn', center=0,
    vmin=-20, vmax=20,
    linewidths=0.5, linecolor='white',
    ax=ax,
    cbar_kws={'label': 'Δ Accuracy (percentage points)'}
)
ax.set_xlabel('Generator', fontweight='bold', fontsize=10)
ax.set_ylabel('Solver', fontweight='bold', fontsize=10)
#ax.set_title('Generator Effect: Delta vs Solver Baseline\n(green = generator makes puzzles easier for this solver)',
#             fontweight='bold', fontsize=13)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'heatmap_delta_vs_baseline.png'), dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Per-Generator Difficulty — Are Some Generators Universally Easier?

In [ ]:
# Average accuracy across ALL solvers per generator
gen_difficulty = merged.groupby('generator_model')['correct'].agg(['mean', 'std', 'count'])
gen_difficulty['mean_%'] = gen_difficulty['mean'] * 100
gen_difficulty['std_%']  = gen_difficulty['std']  * 100
gen_difficulty = gen_difficulty.sort_values('mean_%', ascending=False)

print('Generator difficulty (avg solve rate across all solvers):')
print(gen_difficulty[['mean_%', 'std_%', 'count']].round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x      = np.arange(len(gen_difficulty))
means  = gen_difficulty['mean_%'].values
stds   = gen_difficulty['std_%'].values

norm   = plt.Normalize(vmin=means.min(), vmax=means.max())
colors = [plt.cm.Blues(norm(v) * 0.6 + 0.3) for v in means]

bars = ax.bar(x, means, color=colors, edgecolor='#444444', linewidth=0.6,
              yerr=stds, capsize=4, error_kw={'linewidth': 1, 'color': '#555'})
ax.bar_label(bars, fmt='%.1f%%', padding=5, fontsize=9, color='#333')

ax.set_xticks(x)
ax.set_xticklabels(gen_difficulty.index, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Avg Solve Rate Across All Solvers (%)', fontweight='bold')
ax.set_title('Generator Difficulty\n(higher = generator produces easier-to-solve adaptations)',
             fontweight='bold')
ax.set_ylim(0, 110)
ax.axhline(y=20, color='grey', linestyle='--', alpha=0.4, label='Random (20%)')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'generator_difficulty.png'), dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ------------------------------------------------------------------
# Generator difficulty summary
# Mean solver accuracy (equal number of puzzles per solver)
# ------------------------------------------------------------------

# Accuracy for each (generator, solver) pair
solver_acc = (
    merged
    .groupby(["generator_model", "model_name"])["correct"]
    .mean()
    .reset_index(name="solver_accuracy")
)

# Overall mean across all generators and solvers
overall_mean = solver_acc["solver_accuracy"].mean() * 100

# Aggregate by generator
gen_table = (
    solver_acc
    .groupby("generator_model")
    .agg(
        Mean_Accuracy=("solver_accuracy", lambda x: x.mean() * 100),
        SD=("solver_accuracy", lambda x: x.std() * 100),
        Solvers=("solver_accuracy", "count"),
    )
)

gen_table["Δ vs Overall"] = gen_table["Mean_Accuracy"] - overall_mean

gen_table["Rank"] = (
    gen_table["Mean_Accuracy"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

gen_table = (
    gen_table
    .sort_values("Mean_Accuracy", ascending=False)
    .round({
        "Mean_Accuracy": 1,
        "SD": 1,
        "Δ vs Overall": 1,
    })
)

print(gen_table)

gen_table.to_csv(
    os.path.join(OUTDIR, "generator_difficulty_table.csv")
)

Caption

Table X. Mean solver accuracy for adaptations produced by each generator model. For each generator, accuracy was first computed separately for each solver and then averaged across all solvers. Positive Δ values indicate generators whose adaptations were easier to solve than the overall average across generators, while negative values indicate relatively harder adaptations. Because every solver evaluated the same number of adapted puzzles, this mean solver accuracy is numerically equivalent to the pooled overall accuracy.

## 9. Statistical Test — Generator Effect Significance

In [ ]:
# For each generator, test if its solve rate is significantly different from all others
# Using Mann-Whitney U (non-parametric, correct=binary)
generators = merged['generator_model'].dropna().unique()

print('Mann-Whitney U test: each generator vs all others')
print(f'{"Generator":<35} {"mean_%":>8} {"U-stat":>10} {"p-value":>10} {"sig":>5}')
print('-' * 75)

for gen in sorted(generators):
    group_a = merged[merged['generator_model'] == gen]['correct'].astype(int)
    group_b = merged[merged['generator_model'] != gen]['correct'].astype(int)
    u, p    = mannwhitneyu(group_a, group_b, alternative='two-sided')
    sig     = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'{gen:<35} {group_a.mean()*100:>8.1f} {u:>10.0f} {p:>10.4f} {sig:>5}')

## 10. Summary — Key Findings

In [ ]:
print('='*60)
print('GENERATOR-SOLVER INTERACTION SUMMARY')
print('='*60)

print('\n1. Generator difficulty (avg solve rate across all solvers):')
print(gen_difficulty[['mean_%']].round(1).to_string())

print('\n3. Most beneficial generator-solver pairs (top delta):')
print(delta_df.nlargest(10, 'delta')[['generator', 'solver', 'acc', 'baseline', 'delta', 'n']].round(1).to_string(index=False))

print('\n4. Most harmful generator-solver pairs (bottom delta):')
print(delta_df.nsmallest(10, 'delta')[['generator', 'solver', 'acc', 'baseline', 'delta', 'n']].round(1).to_string(index=False))

In [ ]:
from google.colab import files
import shutil

# Zip the folder
shutil.make_archive('my_outputs', 'zip', '/content/outputs')

# Download
files.download('my_outputs.zip')

Qwen-generated adaptations yielded the highest mean solver accuracy (44.0%), whereas Llama-3.1-generated adaptations yielded the lowest (25.0%). The remaining generators produced similar average accuracies (35.0–38.1%), suggesting limited evidence that any single generator consistently produced universally easier adaptations.

Table X. Mean solver accuracy for puzzles generated by each adaptation model. Accuracy was computed separately for each solver and then averaged across all solvers. Positive Δ values indicate generators whose adaptations were easier than the overall average across generators, while negative values indicate relatively harder adaptations.

# Number of Solver for which

In [ ]:
# Accuracy for each solver × generator pair
solver_gen_acc = (
    merged
    .groupby(["model_name", "generator_model"])["correct"]
    .mean()
    .reset_index()
)

# Best accuracy achieved by each solver
best_per_solver = (
    solver_gen_acc
    .groupby("model_name")["correct"]
    .transform("max")
)

# Keep generators tied for the best accuracy
wins = solver_gen_acc[solver_gen_acc["correct"] == best_per_solver].copy()

# Count wins
win_counts = (
    wins
    .groupby("generator_model")
    .size()
    .rename("Solver Wins")
    .sort_values(ascending=False)
)

print(win_counts)

| Generator  | Mean Solver Accuracy (%) ↑ | Relative Difficulty | Solvers with Highest Accuracy |
| ---------- | -------------------------: | --------------------------------: | ----------------------------: |
| Qwen       |                       44.0 |                              +7.7 |                             5 |
| Claude     |                       38.1 |                              +1.7 |                             1 |
| Gemini-3   |                       37.9 |                              +1.5 |                             3 |
| Gemini-2.5 |                       37.3 |                              +1.0 |                             1 |
| GPT-4o     |                       37.1 |                              +0.8 |                             2 |
| o3         |                       35.0 |                              −1.3 |                             1 |
| Llama-3.1  |                       25.0 |                             −11.3 |                             2 |


Caption

Table X. Difficulty of adapted puzzles grouped by generator model. Mean Solver Accuracy (%) is the average solving accuracy across all solver models for puzzles generated by each generator. Difference from Overall Mean (pp) reports how much easier or harder a generator's adaptations were relative to the overall average accuracy across generators, measured in percentage points (pp). Solvers with Highest Accuracy counts the number of solver models for which a generator produced the highest solving accuracy; ties are counted for all tied generators.

\begin{table}[t]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Generator} &
\textbf{Mean Solver} &
\textbf{Relative} &
\textbf{Solver} \\
&
\textbf{Accuracy (\%)} &
\textbf{Difficulty (pp)} &
\textbf{Wins} \\
\midrule
Qwen-3.5-9B          & \textbf{44.0} & \textbf{+7.7}  & \textbf{5} \\
Claude Haiku 4.5     & 38.1          & +1.7           & 1 \\
Gemini 3 Flash       & 37.9          & +1.5           & 3 \\
Gemini 2.5 Flash     & 37.3          & +1.0           & 1 \\
GPT-4o               & 37.1          & +0.8           & 2 \\
o3                   & 35.0          & $-1.3$         & 1 \\
Llama 3.1 8B         & 25.0          & $\mathbf{-11.3}$ & 2 \\
\bottomrule
\end{tabular}
\caption{Difficulty of adapted puzzles grouped by generator model. \textbf{Mean Solver Accuracy} is the average solving accuracy across all solver models for puzzles generated by each generator. \textbf{Relative Difficulty} reports the difference (in percentage points) between a generator's mean solver accuracy and the overall mean solver accuracy across all generators; positive values indicate relatively easier adaptations, while negative values indicate relatively harder adaptations. \textbf{Solver Wins} denotes the number of solver models for which a generator produced the highest solving accuracy (ties counted for all tied generators).}
\label{tab:generator_difficulty}
\end{table}